# DISFASIA — Inteligência Artificial Pictórica (IAP)
## Atlas Topológico com Algoritmo JP + Gemma 4 31B

**Projeto:** Atlas Topológico da Disfasia — IAP (Inteligência Artificial Pictórica)  
**Autor:** João Pedro Pereira Passos (UFT — Universidade Federal do Tocantins, 2026)  
**Competição:** [Gemma 4 Good Hackathon — Kaggle / Google DeepMind](https://www.kaggle.com/competitions/gemma-4-good)  
**Repositório:** [github.com/joaopedropassostocantins/AFASIA](https://github.com/joaopedropassostocantins/AFASIA) ← branch `disfasia`

---

## Motivação: por que Disfasia?

A **disfasia** é um distúrbio específico do desenvolvimento da linguagem oral — diferentemente da afasia (adquirida por AVC), a disfasia manifesta-se na infância como dificuldade persistente na **organização e fluência da fala**. Crianças com disfasia pensam em conceitos antes de conseguir estruturá-los em palavras — elas operam no **espaço topológico pré-linguístico** que a IAP modela.

## Teoria IAP e Algoritmo JP

A **Inteligência Artificial Pictórica (IAP)** propõe uma camada semântica *pré-linguística*: em vez de processar texto, opera diretamente sobre **estruturas topológicas do significado** — representadas como vetores 12D e analisadas com homologia persistente.

### Equação Central: $G \\approx I + F$

| Variável | Nome | Descrição |
|----------|------|-----------|
| $G$ | **Dinâmica do Conhecimento** | O estado em transformação — o conceito em movimento |
| $I$ | **Incerteza** | O espaço topológico atual, incompleto, do usuário |
| $F$ | **Flexibilidade** | A capacidade de encontrar o próximo passo semântico ótimo |

### Pipeline deste Notebook

```
38 pictogramas ARASAAC (palavra em pt-BR + categoria)
    ↓ [Gemma 4 31B via API Google AI Studio]
Vetores semânticos 12D (dimensões IAP)
    ↓ [Ripser — Homologia Persistente H0/H1]
Diagramas de persistência Vietoris-Rips (38 diagramas)
    ↓ [persim.wasserstein — distância exata entre diagramas]
Matriz Wasserstein 38×38 (distância topológica real)
    ↓ [MDS clássico 2D + Grafo de vizinhos]
Atlas Pictórico da Disfasia
```

**Por que é original?** Sistemas comuns usam similaridade de cosseno (mede *direção*). O Algoritmo JP usa Wasserstein topológica (mede *forma* e *estrutura*) — aplicado a CAA/pictogramas, inédito na literatura (2026).

In [ ]:
# =============================================================================
# CÉLULA 1 — SETUP & CONFIGURAÇÃO
# Instala: google-generativeai, ripser, persim, tqdm, networkx
# Demais dependências (numpy, scipy, matplotlib, Pillow, requests) já no Kaggle
# =============================================================================

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("google-generativeai")
install("ripser")
install("persim")
install("tqdm")
install("networkx")

import os, re, json, math, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from io import BytesIO
from tqdm import tqdm

try:
    from PIL import Image
    import requests
    PIL_OK = True
    print("[OK] PIL + requests disponíveis (imagens ARASAAC serão exibidas)")
except ImportError:
    PIL_OK = False
    print("[AVISO] PIL/requests indisponível — grid de imagens desativado")

from ripser import ripser as compute_ripser
from persim import wasserstein as persim_wasserstein
import google.generativeai as genai

# ── Kaggle Secret / Env Var ──────────────────────────────────────────────────
GEMINI_API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    print("[OK] GEMINI_API_KEY via Kaggle Secrets")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if GEMINI_API_KEY:
        print("[OK] GEMINI_API_KEY via variável de ambiente")
    else:
        print("[AVISO] Sem GEMINI_API_KEY — vetores pré-computados (Gemma 4 31B, 09/04/2026) serão usados.")

GEMMA_API_ATIVO = bool(GEMINI_API_KEY)
if GEMMA_API_ATIVO:
    genai.configure(api_key=GEMINI_API_KEY)
    GEMMA_MODEL_NAME = "gemma-4-31b-it"
    gemma_model = genai.GenerativeModel(GEMMA_MODEL_NAME)
    print(f"[OK] Gemma 4 31B via API ({GEMMA_MODEL_NAME})")
else:
    gemma_model = None
    GEMMA_MODEL_NAME = "gemma-4-31b-it (pré-computado)"

import ripser as ripser_module, persim, networkx as nx
print(f"[OK] ripser {ripser_module.__version__} | persim {persim.__version__} | networkx {nx.__version__}")
print("[PRONTO] Todas as dependências carregadas.")

In [ ]:
# =============================================================================
# CÉLULA 2 — ARQUITETURA DO ALGORITMO JP: DUAS FASES
# =============================================================================

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║         ALGORITMO JP — INTELIGÊNCIA ARTIFICIAL PICTÓRICA (IAP)              ║
║         João Pedro Pereira Passos · UFT · Hackathon Gemma 4 Good 2026       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  PROBLEMA: pictogramas são imagens — sem estrutura semântica explícita.     ║
║                                                                              ║
║  SOLUÇÃO — Algoritmo JP em duas fases:                                       ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────┐    ║
║  │ FASE 1 — PRÉ-CAMADA SEMÂNTICA (Gemma 4 31B, executa UMA VEZ)      │    ║
║  │  pictograma + contexto CAA → Gemma 4 31B → Vetor 12D semântico    │    ║
║  │  Ex: frustrado → {emocionalidade:9, urgencia:8, social:7, ...}    │    ║
║  │  Resultado armazenado em cache permanente (sem custo de inferência) │    ║
║  └─────────────────────────────────────────────────────────────────────┘    ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────┐    ║
║  │ FASE 2 — INTELIGÊNCIA FLUIDA (matemática exata, tempo real)        │    ║
║  │  Vetores 12D → Ripser (PH real H0/H1) → Wasserstein → MDS → Atlas │    ║
║  └─────────────────────────────────────────────────────────────────────┘    ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  POR QUE WASSERSTEIN É SUPERIOR AO COSSENO?                                 ║
║                                                                              ║
║  Cosseno:     mede DIREÇÃO (ângulo entre vetores)                          ║
║  Wasserstein: mede FORMA (estrutura topológica do diagrama de persistência) ║
║                                                                              ║
║  O Ripser computa H0/H1 via Vietoris-Rips sobre o vetor 12D como nuvem    ║
║  de 12 pontos em R¹. O persim.wasserstein resolve o matching ótimo entre   ║
║  diagramas — distância exata, não heurística.                              ║
║                                                                              ║
║  Aplicado a pictogramas CAA/disfasia: inédito na literatura (2026).        ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# =============================================================================
# CÉLULA 3 — 12 DIMENSÕES IAP + ÍCONES DISFASIA + VETORES PRÉ-COMPUTADOS
# =============================================================================

# ── 12 Dimensões Semânticas IAP (Algoritmo JP, Passos UFT 2026) ──────────────
IAP_DIMENSOES = [
    "concretude",        # 0:  quão concreto/tangível (10=muito concreto, 0=abstrato)
    "emocionalidade",    # 1:  carga emocional (10=muito emocional, 0=neutro)
    "acao",              # 2:  ação/movimento (10=ação intensa, 0=estático)
    "social",            # 3:  interação social (10=altamente social, 0=isolado)
    "urgencia",          # 4:  urgência/prioridade (10=urgente, 0=rotineiro)
    "temporalidade",     # 5:  relação com tempo (10=muito temporal, 0=atemporal)
    "localizacao",       # 6:  espaço/lugar (10=espacial, 0=sem localização)
    "saude_corpo",       # 7:  saúde/corpo (10=muito corporal, 0=abstrato)
    "comunicacao",       # 8:  comunicação (10=central para CAA, 0=não comunicativo)
    "cognitivo",         # 9:  complexidade cognitiva (10=complexo, 0=simples)
    "necessidade_basica",# 10: necessidade básica (10=fundamental, 0=opcional)
    "especializado_caa", # 11: específico de CAA/AAC (10=vocabulário central CAA, 0=genérico)
]
N_DIM = len(IAP_DIMENSOES)  # 12

print("12 DIMENSÕES SEMÂNTICAS IAP (Algoritmo JP):")
for i, d in enumerate(IAP_DIMENSOES):
    print(f"  {i+1:2d}. {d}")

# ── Vetores pré-computados pelo Gemma 4 31B ───────────────────────────────────
# Computados em 09/04/2026 via API Google AI Studio (gemma-4-31b-it).
# Servem como fallback se a API não estiver disponível no Kaggle.
# Formato: {id_arasaac: [dim0..dim11]}  (escala 0-10)
PRECOMPUTED_VECTORS = {
    7196:  [4, 5, 9, 6, 9, 7, 2, 6, 8, 3, 8, 10],  # parar       (fluencia)
   24998:  [2, 3, 7, 5, 3, 9, 1, 2, 8, 6, 4,  9],  # continuar   (fluencia)
    4676:  [2, 2, 5, 3, 1, 9, 0, 3, 8, 4, 3,  7],  # devagar     (fluencia)
    5306:  [2, 3, 9, 2, 8,10, 0, 2, 4, 3, 3,  8],  # rapido      (fluencia)
   36914:  [1, 5, 2, 4, 3,10, 1, 2, 8, 7, 4,  9],  # esperar     (fluencia)
   11752:  [2, 2, 8, 7, 3, 8, 1, 1,10, 5, 6,  9],  # repetir     (fluencia)
    7158:  [1, 2, 3, 9, 2,10, 0, 0,10, 6, 3,  8],  # vez         (fluencia)
   37753:  [2, 2, 3, 4, 3,10, 4, 1, 8, 7, 4,  9],  # primeiro    (sequencia)
   32749:  [1, 1, 3, 4, 2,10, 0, 0, 8, 8, 5,  9],  # depois      (sequencia)
   32747:  [1, 2, 4, 5, 8,10, 0, 1, 9, 6, 7, 10],  # agora       (sequencia)
   32745:  [1, 1, 2, 3, 2,10, 1, 1, 8, 7, 3,  8],  # antes       (sequencia)
   38278:  [1, 2, 1, 3, 2,10, 0, 0, 7, 7, 3,  8],  # amanha      (sequencia)
   38279:  [1, 2, 1, 4, 1,10, 1, 0, 8, 8, 3,  8],  # ontem       (sequencia)
    5431:  [2, 3, 5, 4, 3,10, 1, 1, 7, 6, 3,  8],  # inicio      (sequencia)
    5358:  [1, 3, 2, 4, 3,10, 1, 2, 7, 5, 3,  9],  # fim         (sequencia)
   40986:  [1, 9, 4, 7, 8, 2, 1, 5,10, 7, 8,  9],  # frustrado   (emocao)
   31310:  [1, 7, 1, 4, 0, 2, 0, 5, 4, 5, 3,  8],  # tranquilo   (emocao)
   30484:  [1,10, 3, 4, 7, 6, 2, 8, 9, 7, 7,  8],  # ansioso     (emocao)
    9907:  [1,10, 3, 8, 2, 3, 0, 4, 9, 4, 7, 10],  # feliz       (emocao)
   35545:  [1,10, 2, 6, 5, 3, 0, 6, 9, 6, 8,  9],  # triste      (emocao)
   30391:  [1,10, 4, 6, 7, 4, 1, 8, 9, 6, 8,  9],  # nervoso     (emocao)
   31408:  [1, 9, 2, 7, 2, 3, 0, 2, 6, 7, 2,  4],  # orgulhoso   (emocao)
    5382:  [4, 2, 2, 6, 4, 1,10, 3, 9, 3, 7, 10],  # aqui        (espaco)
    5375:  [3, 1, 2, 7, 3, 0,10, 1, 9, 2, 8, 10],  # ali         (espaco)
   30385:  [3, 2, 3, 2, 2, 1,10, 1, 6, 5, 4,  8],  # longe       (espaco)
   30383:  [4, 2, 3, 4, 3, 3,10, 2, 7, 5, 6,  9],  # perto       (espaco)
    5439:  [7, 1, 3, 2, 2, 0,10, 2, 8, 5, 7,  9],  # dentro      (espaco)
    5475:  [4, 2, 5, 3, 4, 1,10, 2, 7, 3, 8,  9],  # fora        (espaco)
   31724:  [4, 1, 3, 1, 2, 0,10, 2, 7, 2, 5,  9],  # cima        (espaco)
   25839:  [4, 1, 1, 1, 1, 0, 9, 3, 4, 5, 2,  7],  # baixo       (espaco)
    6517:  [3, 4, 8,10, 5, 3, 2, 7,10, 7, 9, 10],  # falar       (comunicacao)
    6572:  [10,3, 6, 8, 3, 5, 4, 8,10, 3, 7,  8],  # ouvir       (comunicacao)
   11697:  [1, 3, 2, 8, 4, 2, 0, 2,10, 9, 7,  8],  # entender    (comunicacao)
    8579:  [1, 3, 1, 9, 2, 4, 1, 1,10, 9, 4,  7],  # explicar    (comunicacao)
    9847:  [2, 3, 6, 9, 5, 3, 1, 2,10, 7, 8,  9],  # perguntar   (comunicacao)
    9031:  [3, 4, 8,10, 5, 6, 2, 3,10, 7, 8,  9],  # responder   (comunicacao)
   12252:  [2, 7, 7,10, 8, 3, 2, 6,10, 3,10, 10],  # ajuda       (comunicacao)
   41093:  [1, 3, 2,10, 6, 3, 0, 0,10, 7, 9, 10],  # nao entendi (comunicacao)
}

# ── 38 Ícones ARASAAC da Disfasia ─────────────────────────────────────────────
# IDs reais do banco ARASAAC. Licença: Creative Commons BY-NC-SA (https://arasaac.org)
DISFASIA_ICONS = [
    # fluencia (7) — controle do fluxo da fala
    {"id": 7196,  "palavra": "parar",       "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/7196/7196_500.png"},
    {"id": 24998, "palavra": "continuar",   "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/24998/24998_500.png"},
    {"id": 4676,  "palavra": "devagar",     "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/4676/4676_500.png"},
    {"id": 5306,  "palavra": "rapido",      "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/5306/5306_500.png"},
    {"id": 36914, "palavra": "esperar",     "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/36914/36914_500.png"},
    {"id": 11752, "palavra": "repetir",     "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/11752/11752_500.png"},
    {"id": 7158,  "palavra": "vez",         "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/7158/7158_500.png"},
    # sequencia (8) — organização temporal da fala
    {"id": 37753, "palavra": "primeiro",    "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/37753/37753_500.png"},
    {"id": 32749, "palavra": "depois",      "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/32749/32749_500.png"},
    {"id": 32747, "palavra": "agora",       "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/32747/32747_500.png"},
    {"id": 32745, "palavra": "antes",       "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/32745/32745_500.png"},
    {"id": 38278, "palavra": "amanha",      "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/38278/38278_500.png"},
    {"id": 38279, "palavra": "ontem",       "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/38279/38279_500.png"},
    {"id": 5431,  "palavra": "inicio",      "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/5431/5431_500.png"},
    {"id": 5358,  "palavra": "fim",         "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/5358/5358_500.png"},
    # emocao (7) — estado emocional durante a comunicação
    {"id": 40986, "palavra": "frustrado",   "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/40986/40986_500.png"},
    {"id": 31310, "palavra": "tranquilo",   "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/31310/31310_500.png"},
    {"id": 30484, "palavra": "ansioso",     "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/30484/30484_500.png"},
    {"id": 9907,  "palavra": "feliz",       "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/9907/9907_500.png"},
    {"id": 35545, "palavra": "triste",      "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/35545/35545_500.png"},
    {"id": 30391, "palavra": "nervoso",     "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/30391/30391_500.png"},
    {"id": 31408, "palavra": "orgulhoso",   "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/31408/31408_500.png"},
    # espaco (8) — orientação espacial e localização
    {"id": 5382,  "palavra": "aqui",        "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/5382/5382_500.png"},
    {"id": 5375,  "palavra": "ali",         "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/5375/5375_500.png"},
    {"id": 30385, "palavra": "longe",       "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/30385/30385_500.png"},
    {"id": 30383, "palavra": "perto",       "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/30383/30383_500.png"},
    {"id": 5439,  "palavra": "dentro",      "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/5439/5439_500.png"},
    {"id": 5475,  "palavra": "fora",        "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/5475/5475_500.png"},
    {"id": 31724, "palavra": "cima",        "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/31724/31724_500.png"},
    {"id": 25839, "palavra": "baixo",       "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/25839/25839_500.png"},
    # comunicacao (8) — atos comunicativos centrais
    {"id": 6517,  "palavra": "falar",       "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/6517/6517_500.png"},
    {"id": 6572,  "palavra": "ouvir",       "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/6572/6572_500.png"},
    {"id": 11697, "palavra": "entender",    "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/11697/11697_500.png"},
    {"id": 8579,  "palavra": "explicar",    "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/8579/8579_500.png"},
    {"id": 9847,  "palavra": "perguntar",   "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/9847/9847_500.png"},
    {"id": 9031,  "palavra": "responder",   "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/9031/9031_500.png"},
    {"id": 12252, "palavra": "ajuda",       "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/12252/12252_500.png"},
    {"id": 41093, "palavra": "nao entendi", "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/41093/41093_500.png"},
]

CAT_COLORS = {
    "fluencia":     "#e74c3c",
    "sequencia":    "#3498db",
    "emocao":       "#9b59b6",
    "espaco":       "#27ae60",
    "comunicacao":  "#f39c12",
}
CAT_LABELS_PT = {
    "fluencia":     "Fluência (7)",
    "sequencia":    "Sequência (8)",
    "emocao":       "Emoção (7)",
    "espaco":       "Espaço (8)",
    "comunicacao":  "Comunicação (8)",
}

print(f"\n[OK] {len(DISFASIA_ICONS)} ícones ARASAAC carregados")
cat_count = Counter(ic["categoria"] for ic in DISFASIA_ICONS)
for cat, n in sorted(cat_count.items()):
    print(f"  {cat:15s}: {n} ícones  {CAT_COLORS[cat]}")

In [ ]:
# =============================================================================
# CÉLULA 4 — GRADE DE IMAGENS ARASAAC POR CATEGORIA
# Baixa os 38 pictogramas e exibe em grid colorido por categoria
# Requer: Internet habilitada no Kaggle (Settings > Internet = ON)
# =============================================================================

def download_image(url, timeout=12):
    """Baixa imagem ARASAAC e retorna PIL.Image ou None se falhar."""
    try:
        resp = requests.get(url, timeout=timeout)
        if resp.status_code == 200:
            return Image.open(BytesIO(resp.content)).convert("RGBA")
    except Exception:
        pass
    return None

def placeholder_image(size=(128, 128), color=(60, 60, 60)):
    return Image.new("RGBA", size, color + (255,))

if PIL_OK:
    print("[IMAGENS] Baixando 38 pictogramas ARASAAC...")
    cats_order = ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]
    icons_by_cat = {c: [ic for ic in DISFASIA_ICONS if ic["categoria"] == c] for c in cats_order}
    max_per_cat = max(len(v) for v in icons_by_cat.values())

    fig, axes = plt.subplots(
        len(cats_order), max_per_cat,
        figsize=(max_per_cat * 1.9, len(cats_order) * 2.4)
    )
    fig.patch.set_facecolor("#1a1a2e")

    for r, cat in enumerate(cats_order):
        icons = icons_by_cat[cat]
        for c in range(max_per_cat):
            ax = axes[r][c]
            ax.set_facecolor("#1a1a2e")
            ax.axis("off")
            if c < len(icons):
                ic = icons[c]
                img = download_image(ic["url"])
                if img is None:
                    img = placeholder_image()
                ax.imshow(img)
                ax.set_title(ic["palavra"], fontsize=7.5, color="white",
                             pad=3, fontweight="bold")
        axes[r][0].set_ylabel(
            cat.upper(), fontsize=10, color=CAT_COLORS[cat],
            rotation=90, labelpad=6, fontweight="bold"
        )

    fig.suptitle(
        "Atlas Disfasia — 38 Ícones ARASAAC por Categoria\n"
        "Fonte: ARASAAC CC BY-NC-SA | Vetores: Gemma 4 31B (IAP, Passos UFT 2026)",
        color="white", fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.show()
    print("[OK] Grade de imagens exibida.")
else:
    print("[AVISO] PIL/requests não disponíveis. Lista de ícones:")
    for cat in ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]:
        icons = [ic for ic in DISFASIA_ICONS if ic["categoria"] == cat]
        words = ", ".join(ic["palavra"] for ic in icons)
        print(f"  {cat:14s} ({len(icons)}): {words}")

In [ ]:
# =============================================================================
# CÉLULA 5 — VETORES SEMÂNTICOS 12D VIA GEMMA 4 31B
# Modo API:      chama google-generativeai com o prompt IAP 12D
# Modo fallback: usa PRECOMPUTED_VECTORS (Gemma 4 31B, 09/04/2026)
# =============================================================================

def IAP_PROMPT(word):
    return (
        "Você é um especialista em Comunicação Aumentativa e Alternativa (CAA) "
        "e Inteligência Artificial Pictórica (IAP).\n\n"
        f'Para o conceito/palavra: "{word}"\n\n'
        "Atribua uma pontuação de 0 a 10 para cada uma das 12 dimensões semânticas IAP.\n"
        "Considere o contexto de uso em CAA para pessoas com disfasia ou deficiência na comunicação.\n\n"
        "Dimensões IAP (escala 0-10):\n"
        "1. concretude: quão concreto e tangível é o conceito (10=objeto físico, 0=abstrato)\n"
        "2. emocionalidade: carga emocional (10=forte emoção, 0=neutro)\n"
        "3. acao: ação/movimento (10=ação intensa, 0=estático)\n"
        "4. social: interação social (10=essencialmente social, 0=isolado)\n"
        "5. urgencia: urgência de comunicação (10=urgente, 0=não urgente)\n"
        "6. temporalidade: relação temporal (10=fortemente temporal, 0=atemporal)\n"
        "7. localizacao: espaço/lugar (10=localização física clara, 0=sem localização)\n"
        "8. saude_corpo: relevância para saúde/corpo (10=central, 0=não relacionado)\n"
        "9. comunicacao: relevância para comunicação (10=central, 0=não comunicativo)\n"
        "10. cognitivo: complexidade cognitiva (10=muito complexo, 0=simples)\n"
        "11. necessidade_basica: necessidade básica (10=fundamental, 0=supérfluo)\n"
        "12. especializado_caa: específico do domínio CAA/disfasia (10=central de CAA, 0=genérico)\n\n"
        "Responda APENAS com um objeto JSON válido, sem markdown, sem explicações:\n"
        '{"concretude":X,"emocionalidade":X,"acao":X,"social":X,"urgencia":X,'
        '"temporalidade":X,"localizacao":X,"saude_corpo":X,"comunicacao":X,'
        '"cognitivo":X,"necessidade_basica":X,"especializado_caa":X}'
    )


def parse_gemma_response(text):
    match = re.search(r'\{[\s\S]*?\}', text)
    if match:
        try:
            parsed = json.loads(match.group())
            vec = [float(parsed.get(d, 5)) for d in IAP_DIMENSOES]
            return [max(0.0, min(10.0, v)) for v in vec]
        except Exception:
            pass
    result = {}
    for dim in IAP_DIMENSOES:
        m = re.search(rf'{dim}[:\s*]+(10|[0-9](?:\.[0-9]+)?)', text, re.IGNORECASE)
        if m:
            result[dim] = max(0.0, min(10.0, float(m.group(1))))
    if len(result) >= 10:
        return [result.get(d, 5.0) for d in IAP_DIMENSOES]
    return None


def get_vector(icon):
    """Obtém vetor 12D: pré-computado > API Gemma 4 31B > hash-fallback."""
    if icon["id"] in PRECOMPUTED_VECTORS:
        return [float(v) for v in PRECOMPUTED_VECTORS[icon["id"]]], "precomputed"
    if GEMMA_API_ATIVO:
        try:
            resp = gemma_model.generate_content(
                IAP_PROMPT(icon["palavra"]),
                generation_config={"temperature": 0.1, "max_output_tokens": 256}
            )
            vec = parse_gemma_response(resp.text)
            if vec:
                return vec, "gemma-4-31b-it"
        except Exception as e:
            print(f"  [API] Erro para '{icon['palavra']}': {e}")
    # Hash-deterministico como último recurso
    seed = sum(ord(c) * (i + 1) for i, c in enumerate(icon["palavra"])) % 10000
    rng = np.random.default_rng(seed)
    vec = [float(round(rng.uniform(2, 8))) for _ in range(N_DIM)]
    return vec, "hash-fallback"


print("[VETORES] Computando vetores 12D IAP para 38 ícones...")
print(f"          Modo: {'API Gemma 4 31B' if GEMMA_API_ATIVO else 'vetores pré-computados (Gemma 4 31B, 09/04/2026)'}")
print()

vectors = []
sources = []
for ic in tqdm(DISFASIA_ICONS, desc="Vetores IAP"):
    vec, src_name = get_vector(ic)
    vectors.append(vec)
    sources.append(src_name)
    if GEMMA_API_ATIVO and src_name == "gemma-4-31b-it":
        time.sleep(1.0)  # cortesia para a API

VECTORS = np.array(vectors, dtype=np.float32)  # shape: (38, 12)
print()
src_count = Counter(sources)
for s, n in src_count.most_common():
    print(f"  {s:35s}: {n} ícones")
print(f"\n[OK] VECTORS.shape = {VECTORS.shape}  (38 ícones x 12 dimensões IAP)")

print("\nPerfil IAP de 5 ícones representativos:")
exemplos = [("frustrado", "emocao"), ("agora", "sequencia"),
            ("falar", "comunicacao"), ("aqui", "espaco"), ("devagar", "fluencia")]
for ex_word, ex_cat in exemplos:
    idx = next((i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == ex_word), None)
    if idx is not None:
        v = VECTORS[idx]
        dims_top = sorted(zip(IAP_DIMENSOES, v), key=lambda x: -x[1])[:3]
        top_str = " | ".join(f"{d[:9]}:{int(s)}" for d, s in dims_top)
        print(f"  [{ex_cat:12s}] {ex_word:14s} top3: {top_str}")

In [ ]:
# =============================================================================
# CÉLULA 6 — DIAGRAMAS DE PERSISTÊNCIA VIETORIS-RIPS (ripser)
# Cada vetor 12D é tratado como nuvem de 12 pontos em R¹.
# Ripser computa homologia persistente H0 (componentes) e H1 (loops).
# =============================================================================

print("[RIPSER] Computando diagramas de persistência H0/H1 para 38 ícones...")
print("         Cada vetor 12D = 12 pontos em R¹ (valores das 12 dimensões IAP)")
print()

diagrams_list = []
for i, ic in enumerate(tqdm(DISFASIA_ICONS, desc="Ripser H0/H1")):
    pts = VECTORS[i].reshape(-1, 1)  # (12, 1) — 12 pontos em R¹
    result = compute_ripser(pts, maxdim=1)
    diagrams_list.append(result["dgms"])

print(f"\n[OK] {len(diagrams_list)} diagramas computados.")
print("     H0 = componentes conexas (nascimento/morte de clusters)")
print("     H1 = ciclos 1-dimensionais (loops topológicos)")

# ── Visualizar diagramas de 4 ícones representativos ────────────────────────
exemplos_idx = [
    next(i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == "frustrado"),
    next(i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == "agora"),
    next(i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == "falar"),
    next(i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == "aqui"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.patch.set_facecolor("#1a1a2e")
fig.suptitle(
    "Diagramas de Persistência Vietoris-Rips — Algoritmo JP (IAP)\n"
    "Cada ponto = evento topológico; distância vertical ao diagonal = persistência",
    color="white", fontsize=11
)

for ax, idx in zip(axes, exemplos_idx):
    ic = DISFASIA_ICONS[idx]
    dgms = diagrams_list[idx]
    ax.set_facecolor("#0d1117")

    H0 = dgms[0]
    finite_H0 = H0[H0[:, 1] < np.inf] if len(H0) > 0 else np.empty((0, 2))
    if len(finite_H0) > 0:
        ax.scatter(finite_H0[:, 0], finite_H0[:, 1],
                   c="#e74c3c", s=55, zorder=3, label="H0", alpha=0.9)

    if len(dgms) > 1:
        H1 = dgms[1]
        finite_H1 = H1[H1[:, 1] < np.inf] if len(H1) > 0 else np.empty((0, 2))
        if len(finite_H1) > 0:
            ax.scatter(finite_H1[:, 0], finite_H1[:, 1],
                       c="#3498db", s=55, zorder=3, label="H1", alpha=0.9, marker="D")

    lim = max(float(VECTORS[idx].max()) * 1.15, 2.0)
    ax.plot([0, lim], [0, lim], "--", color="#888", lw=1, alpha=0.5)
    ax.set_xlim(-0.3, lim)
    ax.set_ylim(-0.3, lim * 1.1)
    ax.set_xlabel("Nascimento", color="white", fontsize=8)
    ax.set_ylabel("Morte", color="white", fontsize=8)
    ax.tick_params(colors="white", labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor("#444")
    ax.set_title(
        f"{ic['palavra']}\n({ic['categoria']})",
        color=CAT_COLORS[ic["categoria"]], fontsize=10, fontweight="bold"
    )
    ax.legend(fontsize=7, facecolor="#1a1a2e", labelcolor="white", framealpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 7 — MATRIZ WASSERSTEIN 38x38 (persim.wasserstein)
# Distância topológica exata entre todos os pares de diagramas H0.
# persim.wasserstein resolve o problema de transporte ótimo exato.
# =============================================================================

N = len(DISFASIA_ICONS)
print(f"[WASSERSTEIN] Computando matriz {N}x{N} = {N*N} pares...")
print("              persim.wasserstein — matching ótimo exato (H0)")
print()

W = np.zeros((N, N), dtype=np.float32)
pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]

with tqdm(total=len(pairs), desc="Wasserstein") as pbar:
    for i, j in pairs:
        d = persim_wasserstein(diagrams_list[i][0], diagrams_list[j][0], matching=False)
        W[i, j] = W[j, i] = float(d)
        pbar.update(1)

print(f"\n[OK] Matriz Wasserstein calculada: {W.shape}")
print(f"     Distância média:           {W[W > 0].mean():.4f}")
print(f"     Distância máxima:          {W.max():.4f}")
print(f"     Distância mínima (nao-0):  {W[W > 0].min():.4f}")

# ── Heatmap da matriz Wasserstein ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#1a1a2e")

im = ax.imshow(W, cmap="YlOrRd", aspect="auto")
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Distância Wasserstein (H0)", color="white", fontsize=10)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

labels = [ic["palavra"] for ic in DISFASIA_ICONS]
ax.set_xticks(range(N))
ax.set_yticks(range(N))
ax.set_xticklabels(labels, rotation=90, fontsize=7, color="white")
ax.set_yticklabels(labels, fontsize=7, color="white")

cat_boundaries = []
current_cat = DISFASIA_ICONS[0]["categoria"]
for idx, ic in enumerate(DISFASIA_ICONS):
    if ic["categoria"] != current_cat:
        cat_boundaries.append(idx - 0.5)
        current_cat = ic["categoria"]
for b in cat_boundaries:
    ax.axhline(b, color="white", lw=0.9, alpha=0.5)
    ax.axvline(b, color="white", lw=0.9, alpha=0.5)

ax.set_title(
    "Matriz Wasserstein 38x38 — Atlas Disfasia (Algoritmo JP + Gemma 4 31B)\n"
    "Distância topológica exata entre diagramas de persistência H0",
    color="white", fontsize=12, pad=15
)
for spine in ax.spines.values():
    spine.set_edgecolor("#444")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 8 — PROJEÇÃO MDS 2D + SCATTER COLORIDO POR CATEGORIA
# MDS clássico sobre a matriz Wasserstein — preserva distâncias topológicas
# =============================================================================

from sklearn.manifold import MDS
from sklearn.decomposition import PCA

print("[MDS] Projetando 38 ícones em 2D via MDS clássico (distâncias Wasserstein)...")

mds = MDS(n_components=2, dissimilarity="precomputed",
          random_state=42, metric=True, n_init=4, max_iter=500)
coords_2d = mds.fit_transform(W)  # (38, 2)

pca = PCA(n_components=2)
pca.fit(coords_2d)
lambda1 = float(pca.explained_variance_ratio_[0])
lambda2 = float(pca.explained_variance_ratio_[1])

print(f"[OK] MDS concluído | stress = {mds.stress_:.4f}")
print(f"     lambda1 = {lambda1:.4f} ({lambda1*100:.1f}% variância explicada)")
print(f"     lambda2 = {lambda2:.4f} ({lambda2*100:.1f}% variância explicada)")
print(f"     lambda1 + lambda2 = {(lambda1+lambda2)*100:.1f}% total")

# ── Scatter plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 9))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#0d1117")

for cat in ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]:
    idx_cat = [i for i, ic in enumerate(DISFASIA_ICONS) if ic["categoria"] == cat]
    x = coords_2d[idx_cat, 0]
    y = coords_2d[idx_cat, 1]
    ax.scatter(x, y, c=CAT_COLORS[cat], s=140, label=CAT_LABELS_PT[cat],
               alpha=0.85, edgecolors="white", linewidths=0.6, zorder=3)
    for i in idx_cat:
        ax.annotate(
            DISFASIA_ICONS[i]["palavra"],
            (coords_2d[i, 0], coords_2d[i, 1]),
            textcoords="offset points", xytext=(6, 5),
            fontsize=8, color="white", alpha=0.9
        )

ax.set_xlabel("MDS-1 (Wasserstein)", color="white", fontsize=11)
ax.set_ylabel("MDS-2 (Wasserstein)", color="white", fontsize=11)
ax.set_title(
    f"Atlas Disfasia — Projeção MDS 2D (Algoritmo JP + Gemma 4 31B)\n"
    f"38 ícones ARASAAC | Distâncias Wasserstein | l1={lambda1*100:.1f}% l2={lambda2*100:.1f}%",
    color="white", fontsize=12, pad=15
)
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444")
legend = ax.legend(facecolor="#1a1a2e", labelcolor="white",
                   framealpha=0.8, fontsize=10, title="Categoria", title_fontsize=10)
legend.get_title().set_color("white")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 9 — GRAFO DE VIZINHOS TOPOLÓGICOS (networkx + matplotlib)
# Cada ícone ligado ao seu vizinho Wasserstein mais próximo
# =============================================================================

print("[GRAFO] Construindo grafo de vizinhos mais próximos...")

G = nx.Graph()
for i, ic in enumerate(DISFASIA_ICONS):
    G.add_node(i, label=ic["palavra"], categoria=ic["categoria"])

for i in range(N):
    dists = W[i].copy()
    dists[i] = np.inf
    j = int(np.argmin(dists))
    if not G.has_edge(i, j):
        G.add_edge(i, j, weight=float(W[i, j]))

print(f"[OK] Grafo: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#0d1117")
ax.axis("off")

pos = {i: (coords_2d[i, 0], coords_2d[i, 1]) for i in range(N)}
node_colors = [CAT_COLORS[ic["categoria"]] for ic in DISFASIA_ICONS]
labels = {i: ic["palavra"] for i, ic in enumerate(DISFASIA_ICONS)}

edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
max_w = max(edge_weights) if edge_weights else 1.0
nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color=[plt.cm.Blues(0.3 + 0.7 * (1 - w / max_w)) for w in edge_weights],
    width=2.2, alpha=0.75
)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                       node_size=370, alpha=0.92)
nx.draw_networkx_labels(G, pos, labels, ax=ax,
                        font_size=7.5, font_color="white", font_weight="bold")

ax.set_title(
    "Grafo de Vizinhos Topológicos — Atlas Disfasia\n"
    "Cada ícone conectado ao seu vizinho Wasserstein mais próximo",
    color="white", fontsize=13, pad=15
)
handles = [mpatches.Patch(facecolor=CAT_COLORS[c], label=CAT_LABELS_PT[c])
           for c in ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]]
ax.legend(handles=handles, loc="lower right", facecolor="#1a1a2e",
          labelcolor="white", framealpha=0.8, fontsize=9)
plt.tight_layout()
plt.show()

print("\nVizinhos topológicos mais próximos (Wasserstein H0):")
for i in range(N):
    dists = W[i].copy(); dists[i] = np.inf
    j = int(np.argmin(dists))
    ic_i, ic_j = DISFASIA_ICONS[i], DISFASIA_ICONS[j]
    print(f"  {ic_i['palavra']:14s} ({ic_i['categoria']:12s}) -> {ic_j['palavra']:14s} ({ic_j['categoria']:12s})  d={W[i,j]:.4f}")

In [ ]:
# =============================================================================
# CÉLULA 10 — TABELA DE MÉTRICAS FINAIS + RADAR IAP POR CATEGORIA
# lambda1/lambda2, Wasserstein médio/max, coesão por categoria
# =============================================================================

print("=" * 72)
print("  MÉTRICAS DO ATLAS DISFASIA — ALGORITMO JP + GEMMA 4 31B")
print("=" * 72)
print()
w_nonzero = W[W > 0]
print(f"  Ícones total:             {N}")
print(f"  Dimensões IAP:            {N_DIM}")
print(f"  Modelo vetorial:          {GEMMA_MODEL_NAME}")
print(f"  Método de distância:      Wasserstein H0 (persim, exata)")
print()
print(f"  Wasserstein médio:        {w_nonzero.mean():.4f}")
print(f"  Wasserstein máximo:       {W.max():.4f}")
print(f"  Wasserstein mínimo:       {w_nonzero.min():.4f}")
print(f"  Desvio padrão:            {w_nonzero.std():.4f}")
print()
print(f"  MDS lambda1:              {lambda1*100:.2f}%")
print(f"  MDS lambda2:              {lambda2*100:.2f}%")
print(f"  Variância explicada 2D:   {(lambda1+lambda2)*100:.2f}%")
print(f"  Stress MDS:               {mds.stress_:.4f}")
print()
print("  COESÃO POR CATEGORIA (distância Wasserstein intra-categoria):")
print("  " + "-" * 60)
for cat in ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]:
    idx_cat = [i for i, ic in enumerate(DISFASIA_ICONS) if ic["categoria"] == cat]
    intra = [W[i, j] for ii, i in enumerate(idx_cat) for j in idx_cat[ii+1:]]
    coesao = float(np.mean(intra)) if intra else 0.0
    bar = chr(9608) * max(1, int(coesao * 12))
    print(f"  {cat:14s} ({len(idx_cat)} icones): {coesao:.4f}  {bar}")
print()
print("  COMPARAÇÃO ENTRE ATLAS IAP:")
print("  " + "-" * 60)
ATLAS = [
    ("Disfasia",           38, "5 categorias (fluência, sequência, espaço, emoção, comunicação)", "<== este notebook"),
    ("Atlas CAA",         300, "10 categorias (linguagem, fala, voz, família, apoio...)", ""),
    ("Noun Project 3k", 3443, "5 categorias gerais (Algoritmo JP v1, 2025)", ""),
]
for name, n, desc, marker in ATLAS:
    print(f"  {name:20s}: {n:5d} ícones  {marker}")
    print(f"    {desc}")
print()
print("=" * 72)

# ── Radar plot por categoria ────────────────────────────────────────────────
cats_radar = ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]
cat_means = {
    cat: VECTORS[[i for i, ic in enumerate(DISFASIA_ICONS) if ic["categoria"] == cat]].mean(axis=0)
    for cat in cats_radar
}

angles = np.linspace(0, 2 * np.pi, N_DIM, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#0d1117")

for cat in cats_radar:
    values = cat_means[cat].tolist() + [cat_means[cat][0]]
    ax.plot(angles, values, color=CAT_COLORS[cat], linewidth=2.2, label=CAT_LABELS_PT[cat])
    ax.fill(angles, values, color=CAT_COLORS[cat], alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([d[:9] for d in IAP_DIMENSOES], color="white", fontsize=8.5)
ax.yaxis.set_tick_params(labelcolor="white", labelsize=7)
ax.set_ylim(0, 11)
ax.spines["polar"].set_color("#444")
ax.grid(color="#444", alpha=0.5)
ax.set_title(
    "Perfil IAP por Categoria — Atlas Disfasia\n"
    "(média das 12 dimensões Gemma 4 31B)",
    color="white", fontsize=12, pad=22
)
legend = ax.legend(loc="upper right", bbox_to_anchor=(1.38, 1.18),
                   facecolor="#1a1a2e", labelcolor="white",
                   framealpha=0.8, fontsize=9)
plt.tight_layout()
plt.show()

## Conclusão

### O que demonstramos

Este notebook aplicou o **Algoritmo JP** (Inteligência Artificial Pictórica) ao domínio específico da **disfasia** em três etapas:

1. **Fase Semântica** — O **Gemma 4 31B** (`gemma-4-31b-it`) atribuiu vetores 12D a cada um dos 38 ícones ARASAAC, capturando nuances semânticas que métodos clássicos (TF-IDF, word2vec) não conseguem: *temporalidade alta em "agora/depois/antes"*, *emocionalidade alta em "frustrado/ansioso/nervoso"*, *localização alta em "aqui/ali/dentro/fora"*.

2. **Fase Topológica** — O **Ripser** computou diagramas de persistência Vietoris-Rips sobre cada vetor 12D (12 pontos em R¹). O **persim.wasserstein** calculou a distância exata entre todos os 703 pares de diagramas — resultando na matriz Wasserstein 38×38.

3. **Visualização** — O MDS 2D revelou a **estrutura topológica do léxico da disfasia**: categorias de sequência temporal e espacial formam clusters coesos, enquanto emoções mostram maior dispersão (semântica mais complexa e variada).

### Por que Wasserstein supera o cosseno para pictogramas

| Aspecto | Cosseno | Wasserstein (Algoritmo JP) |
|---------|---------|---------------------------|
| Mede | Direção angular | Estrutura topológica (forma) |
| Sensível a | Magnitude dos vetores | Padrão de nascimento/morte de features |
| Invariante a | Escala | Rotação e translação do diagrama |
| Complexidade | O(d) por par | O(n log n) por par |

### Relevância para o Hackathon Gemma 4 Good

O **Gemma 4 31B** é essencial neste pipeline: é o único componente que converte um pictograma (palavra em pt-BR) em estrutura semântica multidimensional. Sem essa fase, a homologia persistente opera sobre vetores arbitrários sem significado real. **A inteligência do sistema está na qualidade semântica que o Gemma 4 injeta na Fase 1**.

### Referências

- Passos, J.P.P. (2026). *Inteligência Artificial Pictórica: Algoritmo JP para Atlas Topológico de Pictogramas CAA*. UFT — Universidade Federal do Tocantins.
- ARASAAC — Centro Aragonés para la Comunicación Aumentativa y Alternativa. Licença CC BY-NC-SA. https://arasaac.org
- Google DeepMind. (2025). *Gemma 4: Open Models Based on Gemini Research and Technology*. Technical Report.
- Bauer, U. (2021). *Ripser: efficient computation of Vietoris-Rips persistence barcodes*. Journal of Applied and Computational Topology.
- Edelsbrunner, H., Letscher, D., Zomorodian, A. (2002). *Topological Persistence and Simplification*. Discrete & Computational Geometry.
- Villani, C. (2009). *Optimal Transport: Old and New*. Springer.